In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import re
from selenium.webdriver.common.keys import Keys
import time
import pandas as pd
from bs4 import BeautifulSoup

In [117]:
# 1) 브라우저 열기
driver = webdriver.Chrome() #크롬 열기
url = "https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000222833&t_page=%ED%86%B5%ED%95%A9%EA%B2%80%EC%83%89%EA%B2%B0%EA%B3%BC%ED%8E%98%EC%9D%B4%EC%A7%80&t_click=%EA%B2%80%EC%83%89%EC%83%81%ED%92%88%EC%83%81%EC%84%B8&t_search_name=%EC%97%90%EC%8A%A4%ED%8A%B8%EB%9D%BC&t_number=1&dispCatNo=1000001000100150001%2C1000001000800130001&trackingCd=Result_1&tab=review"
driver.get(url) #사이트 접속 

In [ ]:
# 리뷰 버튼 클릭
review_button = driver.find_element(By.XPATH, '//*[@id="main"]/div[2]/div/div[3]/div[2]/div[1]/div/div/button[2]')
review_button.click()

In [ ]:
# shadow root 회피
host1 = driver.find_element(By.TAG_NAME, "oy-review-review-in-product")
root1 = host1.shadow_root
host2 = root1.find_element(By.CSS_SELECTOR, "oy-review-product-review-provider")
host3 = host2.find_element(By.CSS_SELECTOR, "oy-review-review-list-provider")
host4 = host3.find_element(By.CSS_SELECTOR, "oy-review-review-list")
root4 = host4.shadow_root
host5 = root4.find_element(By.CSS_SELECTOR, "oy-review-review-filter")
root5 = host5.shadow_root
host6 = root5.find_element(By.CSS_SELECTOR, "oy-review-review-sort")
root6 = host6.shadow_root

# 최신순 버튼 클릭
buttons = root6.find_elements(By.CSS_SELECTOR, "button.pc-sort-button")
latest_button = buttons[1]
print(latest_button.text)
latest_button.click()

최신순


In [ ]:
# 리뷰 리스트
# final_list = set()

#페이지 끝까지 스크롤
prev_height = driver.execute_script("return document.body.scrollHeight") # 총 스크롤 길이 변수에 할당

while len(final_list) < 2500:
    review_ul = root4.find_element(By.CLASS_NAME, "review-list")
    review_li_list = review_ul.find_elements(By.TAG_NAME, "li")
    for review_li in review_li_list:
        meta_root = review_li.find_element(By.CSS_SELECTOR, "oy-review-review-item").shadow_root
        star_icons = meta_root.find_elements(By.CSS_SELECTOR, ".rating oy-review-star-icon")
        rating = 0 # 점수 초기화
        for star in star_icons:
            star_root = star.shadow_root
            path = star_root.find_element(By.CSS_SELECTOR, "path")
            fill_color = path.get_attribute("fill")
            if fill_color == "#FF5753":
                rating += 1
        date = meta_root.find_element(By.CLASS_NAME, "date").text
        text_root = meta_root.find_element(By.CSS_SELECTOR, "oy-review-review-content").shadow_root
        text = text_root.find_element(By.CLASS_NAME, "content").text
        # print(rating, date, text)
        final_list.add((str(rating), date, text))
    
    # 스크롤 내리기  # 0에서 부터 제일 긴 스크롤까지 해줘
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1.5) # 2초 대기 (stale 에러 방지)

    # 스크롤이 변경됐다면? 새로운 높이 가져오기
    new_height = driver.execute_script("return document.body.scrollHeight") # 총 스크롤 길이
    if new_height == prev_height: # 이전과 같다면 
        break # 멈춤
    prev_height = new_height # 새 스크롤길이로 다시 할당 해줌
    print(len(final_list))
print("현재까지 수집된 리뷰 수: ", len(final_list))

246
255
265
275
286
296
304
315
325
335
345
354
366
375
386
396
405
415
423
434
444
456
465
475
485
495
505
515
525
536
544
554
564
575
585
594
603
613
623
633
642
652
664
673
682
693
703
712
722
732
743
751
761
771
781
791
802
810
821
830
841
850
860
870
882
892
901
912
921
931
941
950
961
970
982
991
1001
1011
1020
1029
1042
1051
1060
1069
1080
1091
1100
1109
1118
1130
1139
1149
1159
1170
1179
1190
1201
1210
1219
1230
1240
1248
1260
1269
1278
1289
1298
1309
1320
1330
1340
1350
1360
1369
1379
1390
1400
1410
1421
1429
1440
1450
1460
1469
1479
1489
1498
1509
1520
1530
1539
1549
1558
1568
1579
1590
1598
1609
1618
1628
1638
1647
1658
1668
1678
1688
1699
1708
1718
1726
1738
1748
1758
1769
1778
1789
1798
1807
1818
1828
1837
1844
1856
1866
1876
1886
1896
1906
1915
1926
1935
1946
1956
1965
1974
1985
1996
2006
2016
2024
2035
2045
2054
2065
2075
2084
2094
2105
2114
2126
2135
2144
2155
2164
2172
2182
2194
2203
2213
2223
2233
2244
2253
2262
2273
2283
2291
2301
2310
2321
2331
2342
2351
2358
2369
2

In [123]:
print(len(final_list))
print(list(final_list)[-1])

2507
('5', '2025.12.30', '넘 순하고 트러블 심했던 제피부에 진정 도움을 주었어요!! ㅎㅎ')


In [ ]:
df = pd.DataFrame(final_list, columns=['별점', '날짜', '리뷰내용'])
df.to_excel('output/oy_review_result.xlsx', index=False)
print(f"총 {len(df)}개 리뷰 저장 완료")

총 2507개 리뷰 저장 완료
